# TOPO_COMPARE — Colab Pipeline

Reproduces results from **"Manifold Matching via Autoencoders"** (ICML submission).

**Runtime:** ~2–3 h on T4 GPU (skip-HPO mode) · ~12 h with full HPO  
**Recommended:** Runtime → Change runtime type → T4 GPU

---
### Steps
1. Mount Google Drive (for persistence across sessions)
2. Install dependencies
3. Configure which experiments to run
4. (Optional) Run hyperparameter optimisation
5. Run final evaluation
6. View results

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Set this to where you uploaded / cloned the repo on Drive ──────────────
REPO_DIR = '/content/drive/MyDrive/TOPO_COMPARE'
# ───────────────────────────────────────────────────────────────────────────

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print('Files found:', os.listdir('.')[:8])

## 2 — Install Dependencies

In [ ]:
%%bash
pip install -q \
    gudhi \
    optuna \
    scanpy \
    anndata \
    umap-learn \
    git+https://github.com/ArGintum/RipserZeros.git

# torchvision is already on Colab; verify versions
python -c "import torch; print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"  
python -c "import gudhi; print('gudhi:', gudhi.__version__)"  
python -c "import optuna; print('optuna:', optuna.__version__)"  

## 3 — GPU Check & Global Config

In [ ]:
import torch, os, sys
sys.path.insert(0, os.getcwd())   # make sure repo modules are importable

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — training will be slow. Change runtime to GPU.')

# ── Global seed (change for different runs) ────────────────────────────────
SEED = 42

# Data cache: keep on Drive so downloads persist across sessions
DATA_RAW = os.path.join(os.getcwd(), 'data', 'raw')
os.makedirs(DATA_RAW, exist_ok=True)
print(f'Data cache: {DATA_RAW}')

## 4 — Experiment Configuration

Edit the variables below to choose which experiments to run.

In [ ]:
# ── Datasets ───────────────────────────────────────────────────────────────
SYNTHETIC_DATASETS = ['spheres', 'swiss_roll', 'concentric_spheres',
                      'linked_tori', 'branching_tree']
REAL_DATASETS      = ['mnist', 'fmnist', 'cifar10']
BIO_DATASETS       = ['pbmc3k', 'paul15']

# Set to True to include each group
RUN_SYNTHETIC = True
RUN_REAL      = True
RUN_BIO       = False   # requires scanpy; set True if installed

# ── Models ─────────────────────────────────────────────────────────────────
MODELS = ['vanilla', 'topoae', 'rtdae', 'mmae']  # add 'geomae', 'ggae' if desired

# ── Latent dimensions ──────────────────────────────────────────────────────
LATENT_DIMS_SYNTH = [2]          # synthetic: visualisation only needs 2D
LATENT_DIMS_REAL  = [2, 16, 32]  # real-world: sweep dimensions

# ── Training ───────────────────────────────────────────────────────────────
SKIP_HPO     = True    # True = use precomputed configs; False = run Optuna HPO
HPO_TRIALS   = 30      # trials per model×dataset (only used if SKIP_HPO=False)
HPO_EPOCHS   = 30
EVAL_EPOCHS  = 100
N_SEEDS      = 3       # seeds for final evaluation (3 is enough for Colab)

print('Configuration saved.')

## 5 — (Optional) Hyperparameter Optimisation

Skip this section if `SKIP_HPO = True` above. Best configs are read from `configs/best_hparams/`.

In [ ]:
import subprocess, sys

if SKIP_HPO:
    print('SKIP_HPO=True — skipping HPO, using precomputed configs.')
else:
    datasets = []
    if RUN_SYNTHETIC: datasets += SYNTHETIC_DATASETS
    if RUN_REAL:      datasets += REAL_DATASETS
    if RUN_BIO:       datasets += BIO_DATASETS

    for dataset in datasets:
        dims = LATENT_DIMS_SYNTH if dataset in SYNTHETIC_DATASETS else LATENT_DIMS_REAL
        for model in MODELS:
            dim_args = ' '.join(str(d) for d in dims)
            print(f'\n>>> HPO: {model} on {dataset} (dims={dims})')
            cmd = [
                sys.executable, 'hyperparam_search_optuna.py',
                '--model',       model,
                '--dataset',     dataset,
                '--latent_dims', *[str(d) for d in dims],
                '--opt_metric',  'density_kl_0_1',
                '--n_trials',    str(HPO_TRIALS),
                '--epochs',      str(HPO_EPOCHS),
                '--device',      DEVICE,
                '--seed',        str(SEED),
                '--results_dir', f'results/hypersearch/{model}_{dataset}',
            ]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout[-2000:] if result.stdout else '')
            if result.returncode != 0:
                print('STDERR:', result.stderr[-1000:])

## 6 — Final Evaluation

In [ ]:
import subprocess, sys

def run_eval(dataset, latent_dim, output_subdir=None):
    out_dir = output_subdir or f'results/final/{dataset}_dim{latent_dim}'
    best_dir = f'configs/best_hparams/{dataset}'
    
    cmd = [
        sys.executable, 'run_final_evaluation.py',
        '--dataset',          dataset,
        '--best_configs_dir', best_dir,
        '--output_dir',       out_dir,
        '--latent_dim',       str(latent_dim),
        '--n_seeds',          str(N_SEEDS),
        '--epochs',           str(EVAL_EPOCHS),
        '--device',           DEVICE,
        '--seed',             str(SEED),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-3000:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    return out_dir

# ── Synthetic datasets (2D latent space) ──────────────────────────────────
if RUN_SYNTHETIC:
    print('='*60, '\nSYNTHETIC DATASETS\n', '='*60)
    for dataset in SYNTHETIC_DATASETS:
        print(f'\n>>> Evaluating {dataset}')
        run_eval(dataset, latent_dim=2)

In [ ]:
# ── Real-world datasets (multiple latent dims) ────────────────────────────
if RUN_REAL:
    print('='*60, '\nREAL-WORLD DATASETS\n', '='*60)
    for dataset in REAL_DATASETS:
        for dim in LATENT_DIMS_REAL:
            print(f'\n>>> Evaluating {dataset} (dim={dim})')
            run_eval(dataset, latent_dim=dim)

In [ ]:
# ── Biological datasets ───────────────────────────────────────────────────
if RUN_BIO:
    print('='*60, '\nBIOLOGICAL DATASETS\n', '='*60)
    for dataset in BIO_DATASETS:
        print(f'\n>>> Evaluating {dataset}')
        run_eval(dataset, latent_dim=2)

## 7 — Quick Single-Run (Debug / Visualise)

Train one model on one dataset and display the 2D latent space.

In [ ]:
DATASET  = 'spheres'
MODEL    = 'mmae'
EPOCHS   = 50
LAT_DIM  = 2

!python run_experiment.py \
    --dataset  {DATASET}  \
    --model    {MODEL}    \
    --latent_dim {LAT_DIM} \
    --epochs   {EPOCHS}   \
    --device   {DEVICE}   \
    --seed     {SEED}     \
    --output_dir results/single_runs

In [ ]:
# Display the saved latent-space plot
import glob, os
from IPython.display import Image, display

plots = sorted(glob.glob(f'results/single_runs/{DATASET}/{MODEL}/**/latent_space.png', recursive=True))
if plots:
    display(Image(plots[-1]))
else:
    print('No plot found — check if run completed successfully.')

## 8 — View Results

In [ ]:
import pandas as pd, glob, json, os

rows = []
for f in glob.glob('results/final/**/summary.csv', recursive=True):
    df = pd.read_csv(f)
    rows.append(df)

if rows:
    summary = pd.concat(rows, ignore_index=True)
    # Show key metrics
    cols = [c for c in summary.columns if c in [
        'dataset', 'model', 'latent_dim',
        'distance_correlation_mean', 'trustworthiness_10_mean',
        'density_kl_0_1_mean', 'wasserstein_H1_mean'
    ]]
    display(summary[cols].sort_values(['dataset','model']))
else:
    print('No summary.csv files found yet. Run evaluation cells first.')

In [ ]:
# Generate LaTeX table rows for the paper
!python generate_latex_table.py \
    --results_dir results/final \
    --output_dir  results/latex

import glob
for f in glob.glob('results/latex/*.tex'):
    print(f'\n--- {os.path.basename(f)} ---')
    with open(f) as fh:
        print(fh.read())